# Portfolio Analytics

Analyses everything the desk holds: HL perps and spot in the configured wallets, the same wallets' tokens on other chains (staked VVV counts as VVV; stablecoins are cash), and IB stock positions read through TWS. Assets split into two groups, each measured against its own benchmark on its own calendar: **Crypto vs BTC** over every day, and **TradFi vs SPY** over US trading days (IB stocks and HL HIP-3 stock, index, commodity and FX perps). Prices come from HL candles, CoinGecko, DefiLlama (wallet tokens, by contract) and IB (adjusted closes, and SPY). TWS must be running for IB positions; without it they are skipped and saved prices are used where present.

The notebook then optimizes a constrained long/short Kelly portfolio, compares it with BTC and SPY, fits a factor model, and saves the results. It never places trades.

## Load implementation

Set the wallet addresses to analyze, then load the companion implementation and create the analytics object.


In [1]:
import sys
from importlib import reload
sys.path.insert(0, '..')

import factor_model, portfolio_risk, portfolio_analytics_implementation
from desk.reloading import reload_desk
# Pick up code changes without restarting the kernel: every desk module, then the research helpers
# and the module that imports them.
reload_desk(factor_model, portfolio_risk, portfolio_analytics_implementation)
from portfolio_analytics_implementation import PortfolioAnalytics

from desk.accounts import hl_wallets

WALLET_ADDRESSES = hl_wallets()  # every HL wallet in accounts.json (full addresses live only there)

# Assets with less price history stay in the benchmark table but not in Kelly or factor sections.
MIN_HISTORY_DAYS = 180
# Years of daily prices to build (CoinGecko gives one; HL candles and DefiLlama the rest).
HISTORY_YEARS = 5

analysis=PortfolioAnalytics(data_dir='data', wallet_addresses=WALLET_ADDRESSES, min_history_days=MIN_HISTORY_DAYS, history_years=HISTORY_YEARS)


## Highlights

What stands out, what needs attention, and what changed since the last run, across every section below. This cell reserves the space; the `show_highlights()` cell near the end fills it in once the sections have run (Run All does both).

In [2]:
analysis.highlights_panel()

Universe,7 Crypto vs BTC; 7 with 180+ days enter Kelly and factor sections.
Crypto exposure,Highest beta to BTC: VVV 1.24; least correlated: HYPE (0.15).
Factor exposures,"Significant: BTC market excess 0.12 (t 2.8), BTC realized vol 0.00 (t 2.0), Alt market - BTC 0.18 (t 8.8)."
Universe change,"Dropped: GOOGL, INIT, INJ, KYIV, META, MU, NET, NVDA, SWMR."
Very volatile,VVV 151%/yr.
Kelly portfolio,"Sharpe 3.49, beta (btc) -0.00, max drawdown -11.4%. In-sample: the weights were fit on the same days."
Out-of-sample alpha,19%/yr (t 0.54): not established: alpha is positive but not distinguishable from zero.
Kelly model portfolio,"Alpha 117%/yr (t 4.33), R² 0%, 417 days to 2026-11-07. In-sample, not a measure of skill."
IB book (actual),"Alpha -35%/yr (t -1.02), R² 13%, 218 days to 2026-11-03; significant: BTC market excess 0.22."
HL book (actual),"Current regime (Short alts, long TradFi, levered, since 2026-10-12): 31 days of returns, 40 days needed; the style model stays empty until then."
Factor data,Ken French data ends 2026-09-30; the last 42 days are not in the style models until its next monthly update.


## Market data refresh

Discovers current nonzero perp positions and spot balances. Perp prices come from HL daily candles (including HIP-3 markets); CoinGecko supplies spot-only prices and market-cap/volume data without requiring an API key. Assets without enough common daily returns against BTC are reported and skipped. Edit `WALLET_ADDRESSES` above to select wallets.


In [3]:
analysis.refresh_market_data()


Refreshed saved prices and market data: HYPE 4 days, BTC 3 days, MELANIA 3 days, TRX 3 days, TRUMP 3 days, xyz:RELIANCE 16 days, DPI 2 days, VVV 2 days. Crypto vs BTC: TRX (HL perp), TRUMP (HL perp), HYPE (HL spot), BTC (HL perp), MELANIA (HL perp), DPI (Wallet), VVV (Wallet). Prices reach back up to 5 years (CoinGecko covers the latest 365 days and all market-cap and volume data; HL candles and DefiLlama extend prices further).
Treating stablecoin balance(s) as cash, not risk assets: USDC.
IB positions unavailable (No module named 'ib_async'); saved IB prices are used where present.
Skipping spot-only wallet asset(s) without a unique CoinGecko ID: MAX (ambiguous CoinGecko symbol).
Using HL candles for perp asset(s) without a CoinGecko mapping: TRX, TRUMP, xyz:RELIANCE.
Skipping asset(s) without at least 30 common returns: xyz:RELIANCE (9 returns with SPY).


## Price history

Loads assets discovered from wallet positions and spot balances, plus BTC for benchmark comparisons.


In [4]:
analysis.load_price_history()


## Daily returns and correlations

Calculates the complete daily-return sample and Pearson correlations.


In [5]:
analysis.calculate_returns()


Combined book: 417 calendar days from 2025-02-13 to 2026-11-07, annualized over 365 days.


## Correlation heatmaps

One heatmap per group on its own calendar: Crypto (HL crypto perps and spot, wallet tokens) over every day, TradFi (IB stocks and HL HIP-3 stock, index, commodity and FX perps) over US trading days. When both are held, a third heatmap shows crypto against TradFi on trading days, with 24/7 prices sampled at each trading day's close.

In [6]:
analysis.plot_correlations()


TradFi correlations: no TradFi assets held.


## Benchmark exposures

Each asset against its own benchmark: crypto against BTC (365-day annualization), TradFi against SPY (252-day). Correlation, beta with a 90% bootstrap range, alpha, volatility and Sharpe use each asset's full pairwise history, not just the dates every asset shares.

In [7]:
analysis.show_benchmark_exposures()

Crypto assets are measured against BTC over every day (365-day annualization); TradFi assets against SPY over US trading days (252-day). Beta ranges come from 2,000 bootstrap resamples. "Corr to other benchmark" is crypto vs SPY or TradFi vs BTC, on US trading days.


,Group,Source,Benchmark,Returns,Correlation,Beta,Beta 90% range,Alpha (/yr),Volatility (/yr),Sharpe,Corr to other benchmark,In Kelly
Asset,,,,,,,,,,,,
VVV,Crypto,Wallet,BTC,496,0.35,1.24,0.99 to 1.50,277.9%,150.6%,1.72,N/A,yes
BTC,Crypto,HL perp,BTC,1368,1.00,1.00,benchmark itself,0.0%,53.0%,0.42,N/A,yes
MELANIA,Crypto,HL perp,BTC,534,0.44,0.99,0.83 to 1.15,-89.7%,100.5%,-1.08,N/A,yes
DPI,Crypto,Wallet,BTC,1103,0.49,0.93,0.82 to 1.04,-17.6%,97.0%,0.04,N/A,yes
TRUMP,Crypto,HL perp,BTC,534,0.39,0.92,0.73 to 1.12,-50.0%,106.3%,-0.64,N/A,yes
HYPE,Crypto,HL spot,BTC,459,0.15,0.35,0.12 to 0.61,192.0%,106.1%,1.75,N/A,yes
TRX,Crypto,HL perp,BTC,1368,0.19,0.24,0.17 to 0.31,34.5%,68.4%,0.58,N/A,yes


## Constrained long/short full-Kelly portfolio

Optimizes signed asset weights from historical returns; the remaining net balance is held as cash. Long/short signs are chosen by the optimizer and are not forced to match current wallet position sides.


In [8]:
analysis.optimize_portfolio()


Gross long exposure: 60.0%; gross short exposure: 45.0%; net exposure: 15.0%.
Expected price PnL is a historical mean-forward-return contribution per $100 of portfolio equity; it excludes funding, fees, and slippage.


,Weight (%),Expected Price PnL / $100 equity (5d)
TRX,15.00,$0.071
HYPE,15.00,$0.403
VVV,15.00,$0.379
DPI,15.00,$-0.012
TRUMP,-15.00,$0.278
MELANIA,-15.00,$0.350
BTC,-15.00,$-0.038
Cash,85.00,$0.057


## Portfolio performance and risk metrics

Charts the Kelly portfolio against BTC, and against SPY when TradFi assets are held. A book with TradFi assets uses US trading days and 252-day annualization; a crypto-only book uses every day and 365. Beta is reported to BTC and to SPY, alpha is net of both, and a sleeve table splits the book into its Crypto and TradFi parts.

In [9]:
analysis.analyze_performance()


,Return
1D,0.69%
7D,4.43%
M (1 month),7.81%
Q (3 months),29.55%
Since Inception,276.13%


,Daily return
Average daily return,0.33%
Largest daily gain,6.01%
Largest daily loss,-5.68%


,All-Time High,Minimum Portfolio Value,Max Drawdown,Max Drawdown (%)
Since inception,$376.13,$91.79,-$34.00,-11.41%


,"Sharpe (excess, sqrt 365)","Sharpe (excess, sqrt 252)","Sharpe (raw, sqrt 252)",Sortino (365d),Beta (BTC),Alpha,"Arithmetic Return (365d, linear)",Hist Vol (daily),"Hist Vol (annualized, 365d)",Historical 1D VaR 95%,Historical VaR 95% ($ / $100),Historical 1D VaR 99%,Historical VaR 99% ($ / $100),Bootstrap 1D VaR 95%,Bootstrap VaR 95% ($ / $100),Bootstrap VaR 95% (90% CI low),Bootstrap VaR 95% (90% CI high),Bootstrap 1D VaR 99%,Bootstrap VaR 99% ($ / $100),Bootstrap VaR 99% (90% CI low),Bootstrap VaR 99% (90% CI high),Daily Return Observations,Historical VaR 99% Tail Observations,Risk-Free Rate,Max Drawdown
In-sample: weights fit and evaluated on same window,3.490,2.900,3.021,6.015,-0.001,116.79%,121.70%,1.75%,33.48%,2.25%,$2.25,3.69%,$3.69,2.25%,$2.25,2.00%,2.58%,3.69%,$3.69,2.90%,4.31%,417,5,5.00%,-11.41%


Risk estimates use 417 daily observations with 365-day crypto annualization. Historical 99% VaR has only 5 observed tail observation(s); treat it as a weak tail estimate. Bootstrap VaR resamples observed daily returns, so it cannot exceed the largest loss in the sample; its range shows sampling uncertainty, not a worst case. Arithmetic return is the daily mean multiplied by 365, not a forecast or compounded return. These are in-sample results: Kelly weights were optimized and evaluated on the same return window; use a walk-forward backtest for out-of-sample evidence.


## Factor model

Fits exposures to BTC, BTC momentum (long or short BTC by the sign of its lagged 30-day return), BTC realized volatility (lagged 30-day), a funding spread, CoinGecko cross-sectional crypto factors where available, and SPY when TradFi assets are held. Crypto factors are built from daily data and compounded onto US trading days for a mixed book.

In [10]:
analysis.fit_factor_model()


Factor model: R²=23.8% | adjusted R²=22.6% | annual alpha=92.73% | n=306 days
Single-factor BTC beta=-0.001 | factor-model BTC beta=0.120
BTC realized vol beta is the daily return per one standard deviation of lagged 30-day volatility; alpha is at average volatility. Funding (low - high) is long low-funding and short high-funding held perps, by lagged 7-day average funding. Returns exclude funding payments and trading costs.
CoinGecko cross-sectional factors exclude assets without market-cap history: TRX, TRUMP, DPI. Size, momentum, and turnover spreads omitted: 3 CoinGecko-priced altcoins; at least 4 are needed.


,Beta,HAC t-stat,Factor premium (%/yr),Premium t-stat,Portfolio contribution (%/yr)
BTC market excess,0.120,2.81,-11.48%,-0.23,-1.38%
BTC momentum,-0.005,-0.16,2.70%,0.06,-0.01%
BTC realized vol,0.002,2.02,N/A,N/A,N/A
Funding (low - high),-0.098,-1.95,37.52%,0.80,-3.68%
Alt market - BTC,0.176,8.85,190.80%,2.03,33.64%
Alpha,N/A,3.74,N/A,N/A,92.73%


,Factor model fit
Observations,306.000000
R-squared,0.238500
Adjusted R-squared,0.225808
Annual alpha,0.927325
BTC beta (factor model),0.120480
BTC beta (single factor),-0.001204


## Style-factor model and return attribution

The industry-standard multi-factor regression: the book's daily excess return = alpha + Σ beta × factor + residual. Equity styles from the Ken French data library (market, size, value, quality, investment, 12-1 month momentum, 1-month reversal, developed-ex-US domicile, tech sector) plus a dividend factor (SCHD − SPY), and the crypto block (BTC, alt market, BTC momentum, funding). Alpha is the intercept; specific return (alpha + residual) is the return attributable to selection. The attribution table and chart split the book's return into factor contributions and specific return. French data updates monthly, so the latest weeks join after its next release. Run after `fit_factor_model()`.

In [11]:
analysis.fit_style_model()

Style-factor model, Kelly model portfolio: 3 factors, 417 days (2025-02-13 to 2026-11-07), R²=0.2%. Alpha=116.7%/yr (t=4.33); information ratio 3.49.
Each beta is the book's exposure to a factor; the contribution is beta x the factor's average return. Alpha is the intercept: the part of the average return no factor explains. Specific return is alpha plus each day's residual; it is the return attributable to selection.


,Beta,HAC t-stat,Factor return (/yr),Contribution (/yr)
BTC market excess,-0.008,-0.19,-25.2%,0.2%
BTC momentum,-0.010,-0.20,-12.7%,0.1%
Funding (low - high),-0.027,-0.44,6.7%,-0.2%
Alpha (intercept),N/A,4.33,N/A,116.7%


,Style model
Book,Kelly model portfolio
Periods,417 days
From,2025-02-13
To,2026-11-07
R-squared,0.001687
Adjusted R-squared,-0.005564
Alpha (/yr),1.166696
Alpha t-stat,4.328456
Residual volatility (/yr),0.334468
Information ratio,3.488213


,Return over sample
BTC market excess,0.2%
BTC momentum,0.1%
Funding (low - high),-0.2%
Specific return (alpha + residual),133.3%
Total excess return,133.5%


Crypto-only book on every day: US equity style factors are omitted.
Left out for covering under three quarters of the sample: Alt market - BTC.
With 417 days and 3 factors, treat |t| below about 2 as noise; betas are estimated from the book's returns, not from holdings' characteristics. This is the Kelly model portfolio, fit on these same days, so its alpha is in-sample.


<style model results, Kelly model portfolio: model, summary, attribution, specific>

## Style-factor attribution of the actual books

The same regression on what was actually traded, so the alpha here is the desk's real specific return. **HL book**: every configured wallet combined into one time-weighted index from the saved HL history, so deposits, withdrawals and transfers between wallets are not performance. It is fit on the **current regime only** (detected from the HL book's exposure per leg, as on the desk dashboard; `fit_style_model_hl_book(regime_start='YYYY-MM-DD')` pins it), never pooled with earlier regimes, and stays empty until the regime has 40 days of returns; the next section shows per-regime betas meanwhile. HL keeps detailed history for about a month, so the model runs on weekly returns until 120 consecutive daily returns have been saved, then on daily. **IB book**: daily NAV and deposits from IB Flex once set up (see `desk/venues/ib_flex.py`), otherwise the desk's saved snapshots, which cannot separate deposits from performance. With few periods only the core factors are kept.

In [12]:
analysis.fit_style_model_hl_book()

HL book: current regime (Short alts, long TradFi, levered, since 2026-10-12): 31 days of returns, 40 days needed; the style model stays empty until then. Per-regime betas so far: style_regimes_hl_book().


## HL book exposure and style betas by regime

A single beta over periods with opposite positions is a weighted average of exposures the book never held at once, so it is unusable for risk. This section rebuilds the HL book's **net and gross exposure per leg** (BTC, alts, TradFi) every week from fills, token transfers and today's balances, splits the history into **regimes** by the sign of each leg and whether gross exposure exceeds 1× equity, and then reports:

- **betas per regime** next to the pooled fit (the current regime is also fit on daily returns when it is too short for weekly betas);
- a **regime-interaction model** (separate intercept and betas per regime, pooled Newey-West errors) that tests whether each beta changed, once two regimes have enough weeks;
- **rolling 12-week betas**, so sign flips are visible instead of averaged away.

Regimes before the crypto factors begin (about a year of CoinGecko data) show exposure only. Run after `fit_factor_model()`.

In [13]:
analysis.style_regimes_hl_book()

,Weeks,Net BTC,Net Alts,Net TradFi,Gross BTC,Gross Alts,Gross TradFi,Gross total,Peak gross,Equity (end)
Regime,,,,,,,,,,
"Long alts, long TradFi, levered (May 2025–Jul 2025)",8,0.09,2.03,0.89,0.09,2.03,0.89,3.02,4.35,$844
"Long BTC, long alts, levered (Jul 2025–Aug 2025)",6,2.49,1.61,0.01,2.49,1.61,0.01,4.12,11.79,$275
Long alts (Aug 2025–Oct 2026),59,0.00,0.45,0.00,0.00,0.45,0.00,0.45,1.30,$240
"Short alts, long TradFi, levered (Oct 2026–Nov 2026)",5,-0.08,-0.33,0.30,0.08,0.69,0.39,1.17,1.71,"$5,296"


,Net BTC,Gross BTC,Net Alts,Gross Alts,Net TradFi,Gross TradFi,Net total,Gross total,Regime
2026-08-30 00:00:00+00:00,0.00,0.00,0.59,0.59,0.00,0.00,0.59,0.59,Long alts (Aug 2025–Oct 2026)
2026-09-06 00:00:00+00:00,0.00,0.00,0.60,0.60,0.00,0.00,0.60,0.60,Long alts (Aug 2025–Oct 2026)
2026-09-13 00:00:00+00:00,0.00,0.00,0.57,0.57,0.00,0.00,0.57,0.57,Long alts (Aug 2025–Oct 2026)
2026-09-20 00:00:00+00:00,0.00,0.00,0.62,0.62,0.00,0.00,0.62,0.62,Long alts (Aug 2025–Oct 2026)
2026-09-27 00:00:00+00:00,0.00,0.00,0.76,0.76,0.00,0.00,0.76,0.76,Long alts (Aug 2025–Oct 2026)
2026-10-04 00:00:00+00:00,0.00,0.00,1.30,1.30,0.00,0.00,1.30,1.30,Long alts (Aug 2025–Oct 2026)
2026-10-11 00:00:00+00:00,0.00,0.00,0.47,0.47,0.00,0.00,0.47,0.47,Long alts (Aug 2025–Oct 2026)
2026-10-18 00:00:00+00:00,0.00,0.00,0.24,0.24,0.41,0.41,0.65,0.65,"Short alts, long TradFi, levered (Oct 2026–Nov 2026)"
2026-10-25 00:00:00+00:00,0.00,0.00,0.07,0.18,0.46,0.46,0.53,0.64,"Short alts, long TradFi, levered (Oct 2026–Nov 2026)"
2026-11-01 00:00:00+00:00,-0.01,0.01,-0.55,0.92,0.34,0.50,-0.22,1.43,"Short alts, long TradFi, levered (Oct 2026–Nov 2026)"


Comparable across regimes: the beta a book with 1x gross exposure would have, with leverage changes inside a regime removed period by period, so a sign flip is a change of positioning, not of leverage. Alpha is per 1x gross and is not quoted from under 40 days.


,"Long alts | 41 weeks, 2026-01-04 to 2026-10-11","Short alts, long TradFi, levered, daily | 22 days, 2026-10-13 to 2026-11-03"
,Beta,Beta
BTC market excess,0.058,-1.929
Alt market - BTC,0.686,-0.595
Funding (low - high),0.010,0.626
"Gross exposure (avg, x equity)",0.47,0.94
Gross range (x equity),0.27 to 1.30,0.28 to 2.46
Gross BTC (avg),0.00,0.00
Gross Alts (avg),0.47,0.49
Gross TradFi (avg),0.00,0.45
Observations,41 weeks,22 days


SPY is scaled by TradFi gross, the alt and funding factors by alt gross, BTC by crypto (BTC + alts) gross. Blank where the leg was under 5% of equity: a beta to a factor the book held no position in is correlation through other holdings, not exposure.
Short alts, long TradFi, levered (Oct 2026–Nov 2026): only 3 weeks with factor data, too few for 3 betas.
Regime interaction model not run: it needs two regimes with at least 9 weeks of factor data each.


Before the factors begin (CoinGecko market data covers about a year), so exposure only: Long BTC, long alts, levered (Jul 2025–Aug 2025); Long alts, long TradFi, levered (May 2025–Jul 2025).
Exposure is rebuilt from fills (perps) and from balances, spot fills and transfers (spot tokens); it is the average of daily exposure over each week. A pooled beta mixes the regimes; use the per-regime and interaction betas for risk, and the exposure table for what was actually held.


<HL book regime results: exposure, daily_exposure, betas, betas_delevered, betas_per_leg, fits, interaction, rolling>

In [14]:
analysis.fit_style_model_ib_book()

IB Flex not set up (no Flex token stored); using saved snapshots.
IB book (IB Flex NAV, net of deposits and withdrawals): 264 daily returns; using daily returns: 264.
Style-factor model, IB book (actual): 4 factors, 218 days (2026-01-02 to 2026-11-03), R²=12.8%. Alpha=-35.3%/yr (t=-1.02); information ratio -1.31.
Each beta is the book's exposure to a factor; the contribution is beta x the factor's average return. Alpha is the intercept: the part of the average return no factor explains. Specific return is alpha plus each day's residual; it is the return attributable to selection.


,Beta,HAC t-stat,Factor return (/yr),Contribution (/yr)
BTC market excess,0.218,5.36,-17.0%,-3.7%
Alt market - BTC,0.031,1.07,261.2%,8.0%
BTC momentum,-0.019,-0.74,4.2%,-0.1%
Funding (low - high),0.001,0.03,54.2%,0.0%
Alpha (intercept),N/A,-1.02,N/A,-35.3%


,Style model
Book,IB book (actual)
Periods,218 days
From,2026-01-02
To,2026-11-03
R-squared,0.127934
Adjusted R-squared,0.111557
Alpha (/yr),-0.352886
Alpha t-stat,-1.018919
Residual volatility (/yr),0.270193
Information ratio,-1.306053


,Return over sample
BTC market excess,-2.2%
Alt market - BTC,4.8%
BTC momentum,-0.0%
Funding (low - high),0.0%
Specific return (alpha + residual),-21.1%
Total excess return,-18.6%


Crypto-only book on every day: US equity style factors are omitted.
With 218 days and 4 factors, treat |t| below about 2 as noise; betas are estimated from the book's returns, not from holdings' characteristics.


<style model results, IB book (actual): model, summary, attribution, specific>

## Cross-sectional factor premia

Tests whether characteristics earn a premium across crypto assets, with t-stats that respect correlation across assets. **Fama-MacBeth** runs one cross-sectional regression per day of every asset's return on its standardized, one-day-lagged characteristic, then takes the t-stat from the time series of daily slopes (Newey-West), so a day when all assets move together moves only that day's slope. **Portfolio sorts** give the same test as a long top-half / short bottom-half spread. Characteristics: 30-day momentum, 30-day volatility and 60-day BTC beta from prices (every crypto asset, including short-history ones), size and turnover where CoinGecko has market data, and 7-day funding for perps. A joint regression controls the broadly covered characteristics for each other.

In [15]:
analysis.estimate_factor_premia()

Fama-MacBeth needs at least 8 crypto assets; 7 are held.


## Alpha split-sample test

Splits the factor-model dates in half. Out-of-sample rows fit the Kelly weights on one half and measure factor-adjusted alpha on the other, so the weights never see the dates they are judged on.


In [16]:
analysis.test_alpha_out_of_sample()


First half: 2026-01-02 to 2026-06-03 (153 days). Second half: 2026-06-04 to 2026-11-03 (153 days).
Only the out-of-sample rows test the alpha: their weights never saw the dates they are evaluated on. Every in-sample row reuses dates the weights were fit on.


Forward test (fit on first half, evaluated on second): alpha not established: alpha is positive but not distinguishable from zero.
A factor is dropped from a window when it duplicates another there, such as BTC momentum when BTC stays on one side of its 30-day trend throughout.
Each half has about 153 days and 5 factors, and the weights cover 7 assets, so treat these estimates as very noisy.


## Fill in the highlights

Summarizes the sections above into the Highlights panel at the top of the notebook (shown here instead if the panel cell was not run).


In [17]:
analysis.show_highlights()

Highlights updated at the top of the notebook.


## Save analysis outputs

Writes calculated tables and series to CSV files in `data/`.


In [18]:
analysis.save_outputs()


Saved portfolio, factor, market, and return outputs to data/.
